# CryptoGuard++ · Week 1 — Graph ML Foundations

**Learning goal:** Understand `Data`, `x`, `edge_index`, message passing, tensor shapes, GCN training, and node classification before moving to the Elliptic Bitcoin dataset.

**Execution:** Open in Google Colab → Runtime → Run all, or execute one cell at a time. CPU is enough for Cora. This notebook is an original practice notebook, not a Stanford/PyG official notebook.

### Seven-day roadmap

| Day | Learn | Deliverable |
|---|---|---|
| 1 | Graph concepts and node classification | Explain nodes/edges/features/labels |
| 2 | PyG `Data` and Cora | Inspect tensor shapes |
| 3 | Message passing in plain PyTorch | Implement neighbor aggregation |
| 4 | GCN normalisation | Write GCN from scratch |
| 5 | Node classification | Train PyG GCN on Cora |
| 6 | Baseline and ablation | Compare MLP, GCN and no-edge GCN |
| 7 | Review and report | Save results and explain the experiments |

**References:** [Stanford CS224W Fall 2025](https://snap.stanford.edu/class/cs224w-2025/), [PyG Colabs](https://pytorch-geometric.readthedocs.io/en/stable/get_started/colabs.html), [PyG introduction](https://pytorch-geometric.readthedocs.io/en/stable/get_started/introduction.html), [PyG install](https://pytorch-geometric.readthedocs.io/en/stable/install/installation.html).

## Day 1–2 · Set up PyG and understand a graph

Colab generally has PyTorch installed. Install only PyG. Additional PyG acceleration packages are optional for this simple example.

In [ ]:
%pip -q install torch_geometric


In [ ]:
import random
import copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import remove_self_loops, add_self_loops

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__, '| pyg:', torch_geometric.__version__, '| device:', device)


### Create a tiny graph by hand

`edge_index` shape is `[2, E]`: first row is source nodes, second row is destination nodes. The graph below represents undirected edges 0–1 and 1–2, so each connection appears twice.

In [ ]:
x_toy = torch.tensor([[1.0], [2.0], [4.0]])
edge_toy = torch.tensor([[0, 1, 1, 2],
                         [1, 0, 2, 1]], dtype=torch.long)
toy = Data(x=x_toy, edge_index=edge_toy)
toy.validate(raise_on_error=True)
print(toy)
print('Node features:\n', toy.x)
print('Edge index:\n', toy.edge_index)
print('x shape:', tuple(toy.x.shape), 'edge_index shape:', tuple(toy.edge_index.shape))


## Day 3 · Implement message passing without PyG layers

For each edge `(source, destination)`, add the source node feature to the destination's aggregate. This is an unnormalised SUM aggregator.

In [ ]:
src, dst = edge_toy
messages = x_toy[src]
aggregated = torch.zeros_like(x_toy)
aggregated.index_add_(0, dst, messages)
deg = torch.bincount(dst, minlength=toy.num_nodes)
print('Neighbor SUM for nodes 0, 1, 2:', aggregated.flatten().tolist())
print('Neighbor counts:', deg.tolist())
# Node 0 gets x[1]=2; node 1 gets x[0]+x[2]=1+4=5;
# node 2 gets x[1]=2.
assert aggregated.flatten().tolist() == [2.0, 5.0, 2.0]


## Day 4–5 · Cora dataset and a 2-layer GCN

Cora has 2,708 nodes, 1,433 node features and 7 classes. The default Planetoid public split uses 140 training, 500 validation and 1,000 test nodes. It is a **transductive** benchmark: graph structure and node features are available for all nodes, while only train labels enter training loss. **Do not copy this evaluation protocol to temporal Elliptic.**

In [ ]:
dataset = Planetoid(root='/content/data/Cora', name='Cora')
data = dataset[0]
data.validate(raise_on_error=True)
print(data)
print('x:', tuple(data.x.shape), '| edge_index:', tuple(data.edge_index.shape), '| y:', tuple(data.y.shape))
print('classes:', dataset.num_classes, '| features:', dataset.num_features)
print('train / val / test:', int(data.train_mask.sum()), int(data.val_mask.sum()), int(data.test_mask.sum()))
assert data.x.shape == (2708, 1433)
assert dataset.num_classes == 7
data = data.to(device)


### GCN computation

$$H^{(l+1)} = \sigma\left(\widetilde{D}^{-1/2}\widetilde{A}\widetilde{D}^{-1/2}H^{(l)}W^{(l)}\right).$$

Shape progression for a hidden size of 16: $[2708,1433]\rightarrow[2708,16]\rightarrow[2708,7]$.

In [ ]:
class GCN(nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)

    def forward(self, data):
        x = self.conv1(data.x, data.edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, data.edge_index)
        return x  # logits [num_nodes, num_classes]

model = GCN(dataset.num_features, 16, dataset.num_classes).to(device)
with torch.no_grad():
    output = model(data)
print('Model output:', tuple(output.shape))
assert output.shape == (2708, 7)


### Training loop

Compute cross-entropy loss **only for `train_mask`**. Use validation loss for early stopping and evaluate the test partition after model selection. Scores may vary with seed and runtime.

In [ ]:
def fit_model(model, data, epochs=200, patience=35, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    best_val = float('inf')
    best_state = None
    stale = 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(data)
        train_loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        train_loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            logits = model(data)
            val_loss = F.cross_entropy(logits[data.val_mask], data.y[data.val_mask]).item()
            val_acc = (logits[data.val_mask].argmax(dim=1) == data.y[data.val_mask]).float().mean().item()
        history.append((epoch, train_loss.item(), val_loss, val_acc))
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if verbose and (epoch == 1 or epoch % 20 == 0):
            print(f'epoch={epoch:3d} train_loss={train_loss.item():.4f} val_loss={val_loss:.4f} val_acc={val_acc:.3f}')
        if stale >= patience:
            break
    model.load_state_dict(best_state)
    return history

@torch.no_grad()
def evaluate(model, data, mask):
    model.eval()
    logits = model(data)[mask]
    labels = data.y[mask]
    return (logits.argmax(dim=1) == labels).float().mean().item()

history_gcn = fit_model(model, data)
print(f'GCN validation accuracy: {evaluate(model, data, data.val_mask):.4f}')
print(f'GCN test accuracy:       {evaluate(model, data, data.test_mask):.4f}')


## Day 4 (deeper exercise) · Implement a GCN from scratch

Here is a **pure PyTorch sparse adjacency** implementation of the same GCN principle (without `GCNConv`). For each source→destination edge we store adjacency entry `(destination, source)` so `A @ X` aggregates incoming messages. The graph is undirected in Cora, but the convention matters for directed Elliptic.

In [ ]:
edge_no_loop, _ = remove_self_loops(data.edge_index)
edge_loop, _ = add_self_loops(edge_no_loop, num_nodes=data.num_nodes)
src, dst = edge_loop

deg = torch.bincount(dst, minlength=data.num_nodes).float()
norm = deg[src].clamp_min(1).pow(-0.5) * deg[dst].clamp_min(1).pow(-0.5)
adj = torch.sparse_coo_tensor(
    indices=torch.stack([dst, src]),
    values=norm,
    size=(data.num_nodes, data.num_nodes),
    device=device,
).coalesce()
print('Sparse adjacency shape:', tuple(adj.shape), '| nonzeros:', adj._nnz())

class ScratchGCN(nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes, adjacency):
        super().__init__()
        self.fc1 = nn.Linear(num_features, hidden_dim, bias=False)
        self.fc2 = nn.Linear(hidden_dim, num_classes, bias=False)
        self.b1 = nn.Parameter(torch.zeros(hidden_dim))
        self.b2 = nn.Parameter(torch.zeros(num_classes))
        self.adj = adjacency
    def forward(self, data):
        h = torch.sparse.mm(self.adj, self.fc1(data.x)) + self.b1
        h = F.relu(h)
        h = F.dropout(h, p=0.5, training=self.training)
        return torch.sparse.mm(self.adj, self.fc2(h)) + self.b2

scratch = ScratchGCN(dataset.num_features, 16, dataset.num_classes, adj).to(device)
print('Scratch GCN logits:', tuple(scratch(data).shape))
_ = fit_model(scratch, data, verbose=False)
print('Scratch GCN test accuracy:', round(evaluate(scratch, data, data.test_mask), 4))


## Day 6 · Compare MLP and GCN (ablation)

The MLP uses node features but **never sees edges**. A GCN with empty `edge_index` still adds self-loops inside `GCNConv`, so it is a good architecture-controlled no-neighbor comparison.

In [ ]:
class MLP(nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(num_features, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
    def forward(self, data):
        x = F.relu(self.fc1(data.x))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.fc2(x)

mlp = MLP(dataset.num_features, 16, dataset.num_classes).to(device)
_ = fit_model(mlp, data, verbose=False)

no_edges = data.clone()
no_edges.edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
gcn_no_edges = GCN(dataset.num_features, 16, dataset.num_classes).to(device)
_ = fit_model(gcn_no_edges, no_edges, verbose=False)

results = {
    'GCN (with graph)': evaluate(model, data, data.test_mask),
    'Scratch GCN': evaluate(scratch, data, data.test_mask),
    'MLP (features only)': evaluate(mlp, data, data.test_mask),
    'GCN without neighbor edges': evaluate(gcn_no_edges, no_edges, no_edges.test_mask),
}
for name, accuracy in results.items():
    print(f'{name:33s}  test_acc={accuracy:.4f}')


## Day 7 · Check understanding and summarize

**Answer in your notes:**
1. Why does `edge_index` have shape `[2, E]` rather than `[E, 2]`?
2. Why are many undirected Cora edges stored twice?
3. Explain the difference between node classification and graph classification.
4. Why does a 2-layer GCN generally propagate information up to 2 hops?
5. Why must `train_mask` be used when computing loss?
6. How do MLP and GCN results compare? Does the graph help in this run?
7. Why is a standard transductive Cora split **not** an acceptable future-prediction evaluation for temporal Elliptic?

**Week 1 deliverables:** an executed notebook; tensor-shape table; comparison of MLP, PyG GCN and scratch GCN; a brief 1-page summary; GitHub commit. Next week: Elliptic preprocessing, explicit chronological splits, strong Random Forest/XGBoost baseline.